# Program Starter

In [24]:
import json
from pathlib import Path
import pandas as pd
import yfinance as yf

In [25]:
START_DATE = "2006-01-01"  
END_DATE = "2025-12-31"
INTERVAL = "1mo"

BASE = Path("data")
BASE.mkdir(exist_ok=True)

MARKET_FILE = Path("markets.json")

In [26]:
def load_markets(path: str = "markets.json") -> dict:
    with open(path, "r") as f:
        config = json.load(f)

    markets = config["markets"]

    print(f"Initial investment : ${config['initial_investment']:,}")
    print(f"Monthly investment : ${config['monthly_investment']:,}")
    print(f"Markets loaded     : {len(markets)}")
    
    for name, tickers in markets.items():
        print(f"  {name}: {tickers}")

    return config
    

In [27]:
config = load_markets()
markets = config["markets"]


Initial investment : $1,000
Monthly investment : $100
Markets loaded     : 1
  primary_total_market: ['BND', 'GLD', 'TIP', 'TLT', 'VEU', 'VGT']


In [28]:
def formulate_monthly_prices(tickers, start, end, interval) -> pd.DataFrame:
    if isinstance(tickers, str):
        tickers = [tickers]

    data = yf.download(
        tickers,
        start=start,
        end=end,
        interval=interval,
        auto_adjust=True,
        progress=False,
    )

    # Single ticker  → flat columns (Price, Date)
    # Multiple tickers → MultiIndex (Price Level, Ticker)
    if isinstance(data.columns, pd.MultiIndex):
        price_level = (
            "Close" if "Close" in data.columns.get_level_values(0)
            else "Adj Close"
        )
        data = data[price_level]
    else:
        price_col = "Close" if "Close" in data.columns else "Adj Close"
        data = data[[price_col]]
        data.columns = tickers

    if isinstance(data, pd.Series):
        data = data.to_frame(name=tickers[0])

    return data.dropna(how="all")

In [29]:
formulate_monthly_prices("AAPL", START_DATE, END_DATE, INTERVAL)

Ticker,AAPL
Date,
2006-01-01,2.262507
2006-02-01,2.052166
2006-03-01,1.879280
2006-04-01,2.109096
2006-05-01,1.790889
...,...
2025-08-01,231.435730
2025-09-01,254.145599
2025-10-01,269.855652


In [30]:
# Saves prices and recomputes returns for a named market.

def update_monthly_dataset(name, tickers):
    price_file = BASE / f"{name}_prices_monthly.csv"
    returns_file = BASE / f"{name}_returns_monthly.csv"

    if not price_file.exists():
        print(f"[NEW] Creating dataset for {name}")
        data = formulate_monthly_prices(tickers, START_DATE, END_DATE, INTERVAL)
    else:
        print(f"[UPDATE] Updating dataset for {name}")

        existing = pd.read_csv(price_file, index_col=0, parse_dates=True)
        last_date = existing.index.max()

        # Start from next month to avoid overlap
        start_ts = last_date + pd.offsets.MonthBegin(1)
        end_ts = pd.to_datetime(END_DATE)

        if start_ts > end_ts:
            print(f"[OK]     {name} — already up to date ({last_date.date()})")
            data = existing
        else:
            print(f"[UPDATE] {name} — fetching {start_ts.date()} → {END_DATE}")
            new_data = formulate_monthly_prices(
                tickers,
                start_ts.strftime("%Y-%m-%d"), 
                END_DATE, 
                INTERVAL
                )
            data = pd.concat([existing, new_data])
            data = data[~data.index.duplicated(keep="last")].sort_index()

    # Ensure all tickers exist
    missing = [t for t in tickers if t not in data.columns]
    if missing:
        raise ValueError(f"{name} -> Missing tickers: {missing}")

    # Clean rows where any asset missing
    data = data[tickers]    
    data = data.dropna()

    
    nan_counts = data.isnull().sum()
    if nan_counts.any():
        print(f"  ⚠️  NaNs remaining after dropna: {nan_counts[nan_counts > 0].to_dict()}")

    # Save prices
    data.to_csv(price_file)

    # Recompute returns every time
    rets = data.pct_change().dropna()           # (latest-old)/old * 100
    rets.to_csv(returns_file)

    print(f"         → {len(data)} months  |  {data.index.min().date()} → {data.index.max().date()}")
    print(f"         → prices  : {price_file}")
    print(f"         → returns : {returns_file}")

    return {
        "name"         : name,
        "tickers"      : tickers,
        "months"       : len(data),
        "return_rows"  : len(rets),
        "start"        : data.index.min(),
        "end"          : data.index.max(),
        "price_file"   : price_file,
        "returns_file" : returns_file,
        "data"         : data,
        "rets"         : rets,
    }


In [31]:
results = {}
for name, tickers in markets.items():
    results[name] = update_monthly_dataset(name, tickers)

print("\n Pipelines completed for all markets.\n")

[NEW] Creating dataset for primary_total_market
         → 225 months  |  2007-04-01 → 2025-12-01
         → prices  : data\primary_total_market_prices_monthly.csv
         → returns : data\primary_total_market_returns_monthly.csv

 Pipelines completed for all markets.



In [ ]:
# ── Cell 7: Verification Summary ─────────────────────────────────────────────
# Spot-check every market: shape, date range, NaN counts, return sanity.

print("═" * 65)
print(f"{'VERIFICATION SUMMARY':^65}")
print("═" * 65)

for name, r in results.items():
    data = r["data"]
    rets = r["rets"]

    print(f"\n▸ {name}")
    print(f"  Tickers   : {r['tickers']}")
    print(f"  Price rows: {r['months']}  \nReturn rows: {r['return_rows']}")
    print(f"  Date range: {r['start'].date()} → {r['end'].date()}")

    # NaN check
    nan_p = data.isnull().sum().sum()
    nan_r = rets.isnull().sum().sum()
    print(f"  NaNs      : prices={nan_p}  returns={nan_r}  {'✓' if nan_p+nan_r == 0 else '⚠️ CHECK'}")

    # Return sanity — flag any monthly return beyond ±60% (likely data error)
    extreme = (rets.abs() > 0.60)
    if extreme.any().any():
        flagged = rets[extreme.any(axis=1)]
        print(f"  ⚠️  Extreme returns (>±60%) detected:")
        print(flagged)
    else:
        print(f"  Returns   : no extreme values detected  ✓")

    # Annualized stats preview
    ann_ret = rets.mean() * 12 * 100
    ann_vol = rets.std() * (12 ** 0.5) * 100
    print(f"  Ann. Return (%) :")
    print("    " + "  ".join(f"{c}: {v:.1f}" for c, v in ann_ret.items()))
    print(f"  Ann. Vol (%)    :")
    print("    " + "  ".join(f"{c}: {v:.1f}" for c, v in ann_vol.items()))

print("\n" + "═" * 65)

═════════════════════════════════════════════════════════════════
                      VERIFICATION SUMMARY                       
═════════════════════════════════════════════════════════════════

▸ primary_total_market
  Tickers   : ['BND', 'GLD', 'TIP', 'TLT', 'VEU', 'VGT']
  Price rows: 225   Return rows: 224
  Date range: 2007-04-01 → 2025-12-01
  NaNs      : prices=0  returns=0  ✓
  Returns   : no extreme values detected  ✓
  Ann. Return (%) :
    BND: 3.1  GLD: 11.0  TIP: 3.5  TLT: 4.0  VEU: 6.0  VGT: 16.9
  Ann. Vol (%)    :
    BND: 4.7  GLD: 17.0  TIP: 6.0  TLT: 14.3  VEU: 18.0  VGT: 19.7

═════════════════════════════════════════════════════════════════
